# Tracker PROPIO en tiempo real — estabilización + Kalman + IoU

**Qué hace.** Sigue los objetos que da un detector externo frame a frame, compensando el movimiento del dron, y decide cuáles se mueven de verdad. Es el `propio_kf` de la comparativa (v13/v14): la versión con **Kalman** e IoU, no la de MMD.

**Qué lo distingue de ByteTrack.** ByteTrack predice dónde estará cada objeto con un Kalman de velocidad constante **en la imagen**. Desde un dron eso falla: si la cámara gira o avanza, todo se desplaza en la imagen aunque esté quieto. Aquí, **antes de predecir**, el estado del Kalman de cada track se lleva al frame actual con la homografía `H` de la cámara (**estabilización**). Así el Kalman solo explica el movimiento **propio** del objeto. La estabilización y el Kalman son parte de la clase principal, `TrackerPropio` (celda 3.2).

**Pipeline de un frame** (`TrackerPropio.actualizar`):

| Paso | Qué hace | Celda |
|---|---|---|
| 1. Escala | frame original → resolución de trabajo (960 px) | 2.1 |
| 2. Estabilización | `H` = cómo se movió la cámara desde el frame anterior | 2.2 |
| 3. Entrada | detecciones a trabajo, sin cajas gigantes, + propuestas | 2.3 |
| 4. Predicción | cada track: historial y Kalman al frame actual con `H`; Kalman avanza un frame | 3.1 / 3.2 |
| 5. Asociación | IoU (cajas infladas 3 px) + húngaro, en dos rondas: fuertes, luego débiles | 3.2 |
| 6. Corrección | Kalman corrige; nacen tracks (solo con fuertes); mueren tras 1 s sin detección | 3.2 |
| 7. ¿Se mueve? | desplazamiento estabilizado en 1 s, rectitud, paralaje, histéresis | 4.1 |
| 8. Salida | tracks confirmados en píxeles originales | 4.2 |

**Entra en cada frame:**
- `frame`: imagen BGR a resolución **original** (el tracker la reduce a `work_width` = 960 px).
- `detecciones`: `Nx6 [x1, y1, x2, y2, conf, clase]` en píxeles del frame **original** (`clase` = id entero ≥ 0).

**Sale de cada frame:** lista de `TrackSalida` (solo tracks **confirmados** con detección en este frame): `id`, `caja` (px originales), `conf`, `clase` / `nombre_clase`, `en_movimiento`, `desplazamiento_px`, `paralaje`. Para un benchmark MOT: `como_array(tracks)` → `Nx7 [x1, y1, x2, y2, id, conf, clase]`.

**Llevarlo al otro entorno:** allí ya hay detector, así que solo hacen falta las PARTES 1–4 (o los `.py` de `trackers_tiempo_real/`, que tienen exactamente este mismo código). La PARTE 5 es el ejemplo de bucle: la celda **5.1** es la única que se sustituye por el detector real.

> Por qué IoU y no la MMD de UCMCTrack: se probó (v14 y v14b) y fragmentó más en 9 de 10 videos, incluso con la compuerta inflada.

In [ ]:
# ======================================================================
# IMPORTACIONES
# Qué hace : Carga las librerías (el notebook no importa nada de trackers_tiempo_real/: todo el código está aquí)
# ¿Se toca? : No
# ======================================================================
import time
from collections import defaultdict, deque
from dataclasses import dataclass, field
from pathlib import Path

from scipy.optimize import linear_sum_assignment

import cv2
import numpy as np
import pandas as pd
from IPython.display import Video, display

## PARTE 1 — Configuración (lo único que se ajusta)
Todos los parámetros están agrupados por etiqueta: `[CAMARA]`, `[ENTRADA]`, `[ASOCIACION]`, `[CLASE]`, `[MOVIMIENTO]`, `[PARALAJE]`, `[PROPUESTAS]`.

- **1.1 `Config`**: lo común a los 4 trackers. Los marcados "igual para TODOS" deben ser iguales al comparar trackers.
- **Benchmark MOT:** `use_proposals=False` (las propuestas son cajas sin clase y el benchmark las contaría como falsos positivos).
- Las distancias están en píxeles de la resolución de **trabajo** (960 px), no de la original.
- **1.2 `ConfigPropio`**: lo propio de este tracker. `[ASOCIACION PROPIO]` se puede tocar; `[KALMAN]` no (son los ruidos de ByteTrack/BoT-SORT y no se ajustan con los videos de evaluación).

In [ ]:
# ======================================================================
# 1.1 · Config
# Qué hace : Parámetros comunes a todos los trackers (los del tracker están en la celda 1.2)
# ¿Se toca? : SÍ — pero los marcados "igual para TODOS" deben ser iguales al comparar trackers
# ======================================================================
@dataclass
class Config:
    work_width: int = 960          # ancho al que se estima la H y se hace el tracking (el frame se reduce a este ancho)
    # [CAMARA] movimiento global de la cámara (homografía)
    min_inliers: int = 30          # por debajo, la homografía se considera poco fiable
    ransac_px: float = 1.5         # tolerancia RANSAC: más baja = se ajusta más al plano del suelo
    grid: tuple = (4, 3)           # (cols, filas): reparte los puntos por toda la imagen (evita que domine una zona)
    # [ENTRADA] qué detecciones llegan al tracker (igual para TODOS)
    max_box_area_frac: float = 0.03  # se descartan detecciones con caja mayor a esta fracción del frame
    use_proposals: bool = True     # añadir "propuestas": cajas SIN clase donde algo se mueve y el detector no ve nada
    # [ASOCIACION] umbrales comunes (igual para TODOS: ByteTrack, BoT-SORT y UCMC usan los mismos)
    conf_high: float = 0.35        # detecciones "fuertes" (1ª asociación, crean tracks nuevos)
    conf_low: float = 0.10         # detecciones "débiles" (2ª asociación, estilo ByteTrack)
    max_miss_sec: float = 1.0      # segundos que un track sobrevive sin detección
    min_hits: int = 4              # detecciones para confirmar un track (antes no sale)
    # [CLASE] evita que una detección suelta de baja confianza cambie la etiqueta de un track
    cls_min_conf: float = 0.40     # solo las detecciones con esta confianza votan por una clase
    cls_min_hits: int = 3          # votos mínimos para que el track adopte una clase; si no, queda "desconocido" (-1)
    # [MOVIMIENTO] cuándo un track cuenta como 'en movimiento' (igual para TODOS)
    window_sec: float = 1.0        # ventana temporal para medir el desplazamiento neto
    min_move_px: float = 6.0       # desplazamiento mínimo (en la ventana) para considerar movimiento
    rel_move: float = 0.30         # ...y además >= rel_move * tamaño del objeto (sqrt(ancho*alto)); frena cajas grandes inestables
    adaptive: bool = False         # umbral adaptativo con la población de tracks (solo si la mayoría es estática)
    max_adaptive_px: float = 12.0  # tope del umbral adaptativo
    mad_k: float = 4.0             # umbral adaptativo = mediana + k*MAD de la población de tracks
    on_frames: int = 3             # evaluaciones seguidas "moviéndose" para activar el estado
    off_frames: int = 10           # evaluaciones seguidas "quieto" para desactivarlo
    min_straightness: float = 0.60 # desplazamiento neto / recorrido en la ventana; un objeto real avanza en línea, el ruido deambula
    max_unknown_aspect: float = 6.0  # un track SIN clase más alargado que esto es una estructura (cable, borde), no un objeto
    # [PARALAJE] lo que sobresale del suelo se desplaza respecto a él cuando el dron se traslada
    parallax_filter: bool = True
    parallax_angle_deg: float = 30.0   # el residuo debe ser paralelo al flujo del suelo en ese punto
    parallax_k_min: float = -1.5       # k<0: algo fijo en la IMAGEN (cable que resbala, borde)
    parallax_k_max: float = 3.0        # k>0: paralaje (poste, árbol); un vehículo real tiene k >> 3
    parallax_min_fit: float = 0.6      # R² mínimo del ajuste  residuo(t) ≈ k·flujo_suelo_acumulado(t)
    parallax_min_ground_px: float = 3.0  # flujo NETO del suelo mínimo en la ventana (en hover no se decide)
    parallax_skip: tuple = ("car", "truck", "bus", "motorcycle", "small vehicle", "large vehicle")  # clases que NO se filtran
    # [PROPUESTAS] cambios de imagen sin clase: objetos que el detector no conoce (solo si use_proposals)
    proposal_gap_sec: float = 0.15 # separación temporal entre los frames que se restan
    diff_thr: int = 25             # diferencia de gris mínima para contar un píxel como "cambió"
    min_area: int = 25
    max_area: int = 6000
    dilate: int = 3
    border: int = 15               # se ignora el borde (ahí la imagen alineada no tiene datos)
    max_aspect: float = 5.0
    max_residual: float = 3.5      # diferencia media tras alinear; si se supera, la H falló y no se generan propuestas
    proposal_conf: float = 0.5     # confianza que se le da a una propuesta
    proposal_memory_sec: float = 0.5   # segundos que se recuerdan las detecciones (para no proponer sobre ellas)
    proposal_overlap: float = 0.2      # se descarta la propuesta si esta fracción de su área cae sobre una detección recordada
    max_coherence: float = 0.80    # estructura de una sola orientación (cables, líneas, bordillos) -> no es un objeto


def warp_pts(H, pts):
    """Aplica la homografía H a una lista de puntos (N x 2)."""
    pts = np.asarray(pts, np.float32).reshape(-1, 2)
    if len(pts) == 0:
        return pts
    return cv2.perspectiveTransform(pts.reshape(-1, 1, 2), H).reshape(-1, 2)


def iou_pad(a, b, pad):
    """IoU entre cajas (N x 4 contra M x 4) infladas `pad` px por lado (ayuda con objetos de pocos píxeles)."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    a = a + np.array([-pad, -pad, pad, pad]); b = b + np.array([-pad, -pad, pad, pad])
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + bb[None, :] - inter + 1e-9)

In [ ]:
# ======================================================================
# 1.2 · ConfigPropio
# Qué hace : Parámetros propios de este tracker (los comunes están en Config, celda 1.1)
# ¿Se toca? : SÍ — [ASOCIACION PROPIO]. [KALMAN] no: son los de ByteTrack/BoT-SORT
# ======================================================================
@dataclass
class ConfigPropio(Config):
    # [ASOCIACION PROPIO] cómo se une una detección con un track
    match_iou: float = 0.20        # IoU mínimo (con cajas infladas) para asociar
    box_pad: float = 3.0           # px con que se inflan las cajas para el IoU (objetos pequeños)
    # [KALMAN] valores por defecto de ByteTrack/BoT-SORT. NO ajustarlos con los videos de evaluación
    kf_std_pos: float = 1 / 20     # ruido de posición, como fracción de la altura de la caja
    kf_std_vel: float = 1 / 160    # ruido de velocidad, como fracción de la altura de la caja

## PARTE 2 — Motor compartido (igual para los cuatro trackers)
Lo que va **antes** del tracker. Es idéntico en todos para que la comparación sea justa.

### 2.1 Escala
**Entra:** frame original. **Sale:** frame de trabajo (960 px) y su gris suavizado. También pasa las cajas de ida (detecciones → trabajo) y de vuelta (salida → original).

In [ ]:
# ======================================================================
# 2.1 · Escala
# Qué hace : Pasa el frame original a la resolución de trabajo, y las cajas de ida (detecciones) y vuelta (salida)
# ¿Se toca? : No (work_width en Config)
# ======================================================================
class Escala:
    """Entra: frame BGR original. Sale: frame de trabajo (BGR) y su gris suavizado (lo usan la H y las propuestas).
    El factor se fija con el primer frame; si cambia la resolución del stream, se crea un tracker nuevo."""

    def __init__(self, cfg, forma_frame):
        H0, W0 = forma_frame[:2]
        self.k = cfg.work_width / W0                                     # original -> trabajo
        self.ww, self.wh = cfg.work_width, int(round(H0 * self.k))

    def frame(self, frame_bgr):
        small = cv2.resize(frame_bgr, (self.ww, self.wh), interpolation=cv2.INTER_AREA)
        gray = cv2.GaussianBlur(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), (3, 3), 0)
        return small, gray

    def a_trabajo(self, dets):
        d = np.asarray(dets, np.float32).reshape(-1, 6).copy()
        d[:, :4] *= self.k
        return d

    def a_original(self, caja):
        return np.asarray(caja, np.float32) / self.k

### 2.2 Estabilizador (movimiento de la cámara)
**Entra:** gris del frame actual. **Sale:** `(H, ok)`.

`H` lleva un punto del frame **anterior** al **actual** suponiendo que todo es suelo: un objeto quieto en el suelo queda en `H·(posición anterior)`. Se estima con LK + RANSAC sobre puntos repartidos en una rejilla (ninguna zona domina). `ok=False` si no hay puntos coherentes suficientes (borroso, agua, cambio brusco).

In [ ]:
# ======================================================================
# 2.2 · Estabilizador
# Qué hace : Estima cómo se movió la cámara entre el frame anterior y el actual (homografía H, LK + RANSAC)
# ¿Se toca? : No (parámetros en Config → [CAMARA])
# ======================================================================
class Estabilizador:
    """Entra: gris del frame actual. Sale: (H, ok).
    H lleva un punto del frame ANTERIOR al ACTUAL suponiendo que todo es suelo: un objeto quieto en el suelo queda en
    H·(posición anterior). ok=False si no hay suficientes puntos coherentes (imagen borrosa, agua, cambio brusco):
    en ese caso H = identidad y los historiales se reinician."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.prev = None

    def _puntos(self, gray):
        """Puntos repartidos en una rejilla, para que ninguna zona (techos, pastizal) domine la homografía."""
        h, w = gray.shape
        gx, gy = self.cfg.grid
        pts = []
        for i in range(gy):
            for j in range(gx):
                y0, y1, x0, x1 = i * h // gy, (i + 1) * h // gy, j * w // gx, (j + 1) * w // gx
                p = cv2.goodFeaturesToTrack(gray[y0:y1, x0:x1], 150, 0.01, 8)
                if p is not None:
                    pts.append(p.reshape(-1, 2) + np.array([x0, y0], np.float32))
        return np.vstack(pts).reshape(-1, 1, 2).astype(np.float32) if pts else None

    def paso(self, gray):
        if self.prev is None:                                            # primer frame: no hay movimiento que medir
            self.prev = gray
            return np.eye(3), True
        H, ok = np.eye(3), False
        p0 = self._puntos(self.prev)
        if p0 is not None and len(p0) >= 10:
            p1, st, _ = cv2.calcOpticalFlowPyrLK(self.prev, gray, p0, None, winSize=(21, 21), maxLevel=3)
            good = st.ravel() == 1
            if good.sum() >= 10:
                Hc, inl = cv2.findHomography(p0[good], p1[good], cv2.RANSAC, self.cfg.ransac_px)
                if Hc is not None and inl.sum() >= self.cfg.min_inliers:
                    H, ok = Hc, True
        self.prev = gray
        return H, ok

### 2.3 Entrada de detecciones (+ propuestas)
**Entra:** detecciones del detector externo (píxeles originales). **Sale:** detecciones en píxeles de trabajo, sin cajas gigantes (`max_box_area_frac`), más las **propuestas**.

- `EntradaDetecciones.revisar` avisa con un mensaje claro si las cajas llegan como `[x, y, ancho, alto]`.
- **Propuestas:** diferencia con el frame de hace 0.15 s ya compensado con `H` → cajas **sin clase** (-1) donde algo se mueve y el detector no ve nada. Se descartan las que caen sobre una detección reciente (su estela) y las estructuras de una sola orientación (cables, bordillos).

In [ ]:
# ======================================================================
# 2.3 · EntradaDetecciones (+ Propuestas)
# Qué hace : Revisa y escala las detecciones del detector externo; opcionalmente añade "propuestas" sin clase
# ¿Se toca? : No (parámetros en Config → [ENTRADA], [PROPUESTAS])
# ======================================================================
class Propuestas:
    """Cambios respecto al frame t-gap, ya compensando la cámara con H: cajas SIN clase (clase -1) para cosas que se
    mueven y el detector no conoce. Se descartan las que caen sobre una detección reciente (su estela)."""

    def __init__(self, cfg, fps):
        self.cfg = cfg
        self.gap = max(2, int(round(cfg.proposal_gap_sec * fps)))
        self.grays = deque(maxlen=self.gap + 1)
        self.Hs = deque(maxlen=self.gap)
        self.oks = deque(maxlen=self.gap)
        self.memoria = deque(maxlen=max(1, int(round(cfg.proposal_memory_sec * fps))))   # cajas recientes, una entrada por frame

    def paso(self, gray, H, ok, dets):
        """dets: detecciones del frame (trabajo). Devuelve las propuestas (Nx6) ya filtradas."""
        self._recordar(dets, H)
        pr = self._cambios(gray, H, ok)
        pr = self._fuera_de_memoria(pr)
        if len(pr) and len(dets):
            pr = pr[iou_pad(pr[:, :4], dets[:, :4], 2.0).max(1) < 0.1]
        return pr

    def _recordar(self, dets, H):
        """Lleva las cajas recordadas al frame actual con H y añade las de este frame."""
        self.memoria = deque([warp_pts(H, b.reshape(-1, 2)).reshape(-1, 4) for b in self.memoria], maxlen=self.memoria.maxlen)
        if len(dets):
            self.memoria.append(dets[:, :4].astype(np.float32).copy())

    def _cambios(self, gray, H, ok):
        c = self.cfg
        if len(self.grays) > 0:
            self.Hs.append(H); self.oks.append(ok)
        self.grays.append(gray)
        if len(self.grays) < self.gap + 1 or not all(self.oks):
            return np.zeros((0, 6), np.float32)
        M = np.eye(3)
        for h in self.Hs:                                                # H_t @ ... @ H_{t-gap+1}
            M = h @ M
        prev, curr = self.grays[0], gray
        h_, w_ = curr.shape
        warped = cv2.warpPerspective(prev, M, (w_, h_))
        valid = cv2.erode(cv2.warpPerspective(np.full((h_, w_), 255, np.uint8), M, (w_, h_)), np.ones((11, 11), np.uint8))
        d = cv2.bitwise_and(cv2.absdiff(curr, warped), valid)
        if valid.any() and d[valid > 0].mean() > c.max_residual:
            return np.zeros((0, 6), np.float32)                          # la alineación falló: todo "cambió"
        m = (d > c.diff_thr).astype(np.uint8) * 255
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
        m = cv2.dilate(m, np.ones((c.dilate, c.dilate), np.uint8))
        n, _, s, _ = cv2.connectedComponentsWithStats(m)
        gx = cv2.Sobel(curr, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(curr, cv2.CV_32F, 0, 1, ksize=3)
        out = []
        for k in range(1, n):
            x, y, bw, bh, a = s[k]
            if not (c.min_area <= a <= c.max_area):
                continue
            if x < c.border or y < c.border or x + bw > w_ - c.border or y + bh > h_ - c.border:
                continue
            ar = bw / max(bh, 1)
            if ar > c.max_aspect or ar < 1 / c.max_aspect:
                continue
            sx, sy = gx[y:y + bh, x:x + bw], gy[y:y + bh, x:x + bw]
            jxx, jyy, jxy = float((sx * sx).sum()), float((sy * sy).sum()), float((sx * sy).sum())
            coh = np.sqrt((jxx - jyy) ** 2 + 4 * jxy ** 2) / (jxx + jyy + 1e-6)
            if coh > c.max_coherence:
                continue                                                 # línea/cable/borde, no un objeto
            out.append([x, y, x + bw, y + bh, c.proposal_conf, -1])
        return np.array(out, np.float32).reshape(-1, 6)

    def _fuera_de_memoria(self, pr):
        """Quita las propuestas cuya área cae en >= proposal_overlap sobre alguna detección recordada."""
        if not len(pr) or not self.memoria:
            return pr
        D = np.vstack(self.memoria)
        x1 = np.maximum(pr[:, None, 0], D[None, :, 0]); y1 = np.maximum(pr[:, None, 1], D[None, :, 1])
        x2 = np.minimum(pr[:, None, 2], D[None, :, 2]); y2 = np.minimum(pr[:, None, 3], D[None, :, 3])
        inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
        area = np.maximum((pr[:, 2] - pr[:, 0]) * (pr[:, 3] - pr[:, 1]), 1.0)
        return pr[(inter / area[:, None]).max(1) < self.cfg.proposal_overlap]


class EntradaDetecciones:
    """Entra: detecciones del detector EXTERNO, Nx6 [x1, y1, x2, y2, conf, clase] en píxeles del frame ORIGINAL
    (clase = id entero >= 0). Sale: detecciones en píxeles de TRABAJO, sin cajas gigantes, + propuestas (clase -1)."""

    def __init__(self, cfg, fps):
        self.cfg = cfg
        self.propuestas = Propuestas(cfg, fps) if cfg.use_proposals else None

    @staticmethod
    def revisar(dets):
        """Convierte a Nx6 float32 y avisa con un mensaje claro si el formato no es el esperado."""
        if dets is None:
            return np.zeros((0, 6), np.float32)
        d = np.asarray(dets, np.float32)
        if d.size == 0:
            return np.zeros((0, 6), np.float32)
        if d.ndim != 2 or d.shape[1] < 6:
            raise ValueError(f"Las detecciones deben ser Nx6 [x1, y1, x2, y2, conf, clase]; llegó {d.shape}")
        d = d[:, :6]
        if np.any(d[:, 2] < d[:, 0]) or np.any(d[:, 3] < d[:, 1]):
            raise ValueError("Cajas con x2 < x1 o y2 < y1: ¿el detector devuelve [x, y, ancho, alto]? Hay que pasarlas a esquinas.")
        return d

    def preparar(self, dets, escala, gray, H, ok):
        d = escala.a_trabajo(self.revisar(dets))
        if len(d):
            area = (d[:, 2] - d[:, 0]) * (d[:, 3] - d[:, 1])
            d = d[area <= self.cfg.max_box_area_frac * escala.ww * escala.wh]
        if self.propuestas is not None:
            pr = self.propuestas.paso(gray, H, ok, d)
            if len(pr):
                d = np.vstack([d, pr]) if len(d) else pr
        return d

### 2.4 Track
Objeto `track` común: caja, votos de clase, historial de centros y estado de movimiento.

El historial está siempre **en coordenadas del frame actual**: cada frame se re-proyecta con `H`. Así un objeto quieto tiene un historial quieto aunque el dron se mueva. La clase es la más votada (solo votan detecciones con `conf ≥ 0.40`, y hacen falta 3 votos).

In [ ]:
# ======================================================================
# 2.4 · Track
# Qué hace : Objeto 'track' común: caja, votos de clase, historial ESTABILIZADO y estado de movimiento
# ¿Se toca? : No
# ======================================================================
class Track:
    """El historial de centros (pts) está siempre en coordenadas del frame ACTUAL: cada frame se re-proyecta con H.
    Así un objeto quieto en el suelo tiene un historial quieto aunque el dron se mueva."""

    def __init__(self, tid, det, fidx, maxlen, cls_min_conf=0.40, cls_min_hits=3):
        self.id = tid
        self.box = det[:4].astype(np.float32).copy()
        self.conf = float(det[4])
        self.hits, self.miss = 1, 0
        self.cls_min_conf, self.cls_min_hits = cls_min_conf, cls_min_hits
        self.votes = defaultdict(float)
        self.counts = defaultdict(int)
        self._vote(det)
        self.pts = np.array([self._center()], np.float32)   # historial de centros (frame actual)
        self.gs = np.zeros((1, 2), np.float32)             # flujo que la cámara induce en el track, por frame (paralaje)
        self._g = np.zeros(2, np.float32)
        self.maxlen = maxlen
        self.first = self.last = fidx
        # estado de movimiento (lo llena DecisorMovimiento)
        self.disp, self.vec, self.moving = 0.0, np.zeros(2, np.float32), False
        self.straight, self.par, self.k_par = 1.0, False, None
        self.on_cnt = self.off_cnt = 0
        self.toggles = self.moving_frames = self.par_frames = 0
        self.max_disp = 0.0

    def _center(self):
        return [(self.box[0] + self.box[2]) / 2, (self.box[1] + self.box[3]) / 2]

    def _vote(self, det):
        if det[5] >= 0 and det[4] >= self.cls_min_conf:
            self.votes[int(det[5])] += float(det[4]); self.counts[int(det[5])] += 1

    @property
    def cls(self):
        """Clase más votada (suma de confianzas) si tiene al menos cls_min_hits votos; si no, -1 (desconocido)."""
        if not self.votes:
            return -1
        c = max(self.votes, key=self.votes.get)
        return c if self.counts[c] >= self.cls_min_hits else -1

    def warp(self, H, H_ok=True):
        """Lleva caja e historial al frame actual con H. Sin H fiable, el historial se reinicia (no es comparable)."""
        old_c = np.array(self._center(), np.float32)
        self.box = warp_pts(H, [self.box[:2], self.box[2:]]).reshape(-1).astype(np.float32)
        self._g = np.array(self._center(), np.float32) - old_c          # flujo del suelo en este punto
        self.pts = warp_pts(H, self.pts)
        A = (H[:2, :2] / H[2, 2]).astype(np.float32)
        self.gs = self.gs @ A.T
        if not H_ok:
            self.pts, self.gs = self.pts[-1:], self.gs[-1:]

    def update(self, det, fidx):
        self.box = det[:4].astype(np.float32).copy()
        self.conf = float(det[4])
        self.hits += 1; self.miss = 0; self.last = fidx
        self._vote(det)
        self.pts = np.vstack([self.pts, self._center()])[-self.maxlen:]
        self.gs = np.vstack([self.gs, self._g])[-self.maxlen:]

## PARTE 3 — El tracker propio
### 3.1 Kalman con estabilización (`KalmanXYAH`)
Estado `(x, y, a, h, vx, vy, va, vh)`: centro, aspecto, alto y sus velocidades (px/frame). Mismo modelo y ruidos que ByteTrack; lo único nuevo es **`estabilizar`**:

- el **centro** se lleva con la `H` completa;
- la **velocidad** y la **covarianza**, con la parte lineal `A` de `H`;
- la **altura** de la caja se escala por `√|det A|` (zoom o cambio de altura del dron); el aspecto no cambia.

Ciclo por track: `estabilizar` (cámara) → `predecir` (movimiento propio) → `corregir` (con la detección).

In [ ]:
# ======================================================================
# 3.1 · KalmanXYAH
# Qué hace : Filtro de Kalman de velocidad constante sobre la caja, con la compensación de cámara (warp)
# ¿Se toca? : No
# ======================================================================
def caja_a_xyah(b):
    """[x1,y1,x2,y2] -> [centro x, centro y, ancho/alto, alto] (estado de ByteTrack)."""
    w, h = b[2] - b[0], max(b[3] - b[1], 1e-3)
    return np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2, w / h, h], np.float64)


def xyah_a_caja(z):
    """[centro x, centro y, ancho/alto, alto] -> [x1,y1,x2,y2]."""
    h = max(z[3], 1e-3); w = z[2] * h
    return np.array([z[0] - w / 2, z[1] - h / 2, z[0] + w / 2, z[1] + h / 2], np.float32)


class KalmanXYAH:
    """Estado (x, y, a, h, vx, vy, va, vh): centro, aspecto, alto y sus velocidades (px/frame).
    Mismo modelo y ruidos que ByteTrack/BoT-SORT; lo único nuevo es `estabilizar`."""

    def __init__(self, std_pos=1 / 20, std_vel=1 / 160):
        self.F = np.eye(8); self.F[:4, 4:] = np.eye(4)      # transición: posición += velocidad (1 frame)
        self.Hm = np.eye(4, 8)                               # se observa (x, y, a, h), no la velocidad
        self.wp, self.wv = std_pos, std_vel

    def iniciar(self, z):
        """Estado inicial a partir de la primera detección: velocidad 0 con mucha incertidumbre."""
        h = z[3]
        std = [2 * self.wp * h, 2 * self.wp * h, 1e-2, 2 * self.wp * h,
               10 * self.wv * h, 10 * self.wv * h, 1e-5, 10 * self.wv * h]
        return np.r_[z, np.zeros(4)], np.diag(np.square(std))

    @staticmethod
    def estabilizar(mean, cov, H):
        """Lleva el estado al frame actual con la homografía H (compensación de cámara).
        Centro: H completa. Velocidad y covarianza: parte lineal A. Altura: escala sqrt(|det A|). Aspecto: igual."""
        A = H[:2, :2] / H[2, 2]                              # parte lineal (local) de H
        s = float(np.sqrt(abs(np.linalg.det(A))))            # cambio de escala (zoom / cambio de altura)
        J = np.eye(8)
        J[0:2, 0:2] = A; J[3, 3] = s                         # posición y altura
        J[4:6, 4:6] = A; J[7, 7] = s                         # velocidades
        m = J @ mean
        m[:2] = warp_pts(H, mean[:2])[0]                     # el centro, con la homografía exacta
        return m, J @ cov @ J.T

    def predecir(self, mean, cov):
        """Avanza un frame (movimiento PROPIO del objeto). El ruido escala con la altura de la caja (como ByteTrack)."""
        h = mean[3]
        std = [self.wp * h, self.wp * h, 1e-2, self.wp * h, self.wv * h, self.wv * h, 1e-5, self.wv * h]
        return self.F @ mean, self.F @ cov @ self.F.T + np.diag(np.square(std))

    def corregir(self, mean, cov, z):
        """Corrige el estado con una detección z = (x, y, a, h)."""
        h = mean[3]
        R = np.diag(np.square([self.wp * h, self.wp * h, 1e-1, self.wp * h]))   # ruido de la medida
        S = self.Hm @ cov @ self.Hm.T + R                    # covarianza de la innovación
        K = cov @ self.Hm.T @ np.linalg.inv(S)               # ganancia de Kalman
        return mean + K @ (z - self.Hm @ mean), cov - K @ S @ K.T

### 3.2 `TrackerPropio` (la clase principal)
`actualizar(frame, detecciones)` hace los 8 pasos en orden; `seguir(dets, H, ok)` hace los pasos 4–6 (sirve también para probar el tracker con detecciones ya escaladas):

```
4. _predecir ... track.warp(H)                       historial al frame actual (para el paso 7)
                 kalman.estabilizar(H)               Kalman al frame actual       <- ESTABILIZACIÓN
                 kalman.predecir()                   Kalman un frame adelante     <- movimiento propio
5. _asociar .... IoU caja PREDICHA vs detección (infladas 3 px), húngaro, clase compatible
                 ronda 1: fuertes (>= 0.35) · ronda 2: débiles (0.10-0.35) con los tracks sobrantes
6. _corregir ... Kalman corrige; el track guarda la caja FILTRADA
   _nuevo ...... nace un track por cada detección fuerte sin pareja; muere tras max_miss_sec sin detección
```

In [ ]:
# ======================================================================
# 3.2 · TrackerPropio
# Qué hace : El pipeline completo de un frame (pasos 1-8 del inicio del notebook), en una sola clase
# ¿Se toca? : No (parámetros en ConfigPropio)
# ======================================================================
class TrackerPropio:
    nombre = "propio_kf"

    def __init__(self, fps, cfg=None, nombres_clases=None):
        self.cfg = cfg or ConfigPropio()
        self.fps = float(fps)
        self.nombres_clases = dict(nombres_clases or {})
        c = self.cfg
        # piezas del pipeline
        self.escala = None                                   # se crea con el primer frame (paso 1)
        self.estabilizador = Estabilizador(c)                # paso 2
        self.entrada = EntradaDetecciones(c, fps)            # paso 3
        self.kalman = KalmanXYAH(c.kf_std_pos, c.kf_std_vel)  # pasos 4 y 6
        self.decisor = DecisorMovimiento(c, fps, ids_sin_paralaje(self.nombres_clases, c))   # paso 7
        # estado
        self.tracks, self.nid, self.frame = [], 1, 0
        self.max_miss = max(2, int(round(c.max_miss_sec * fps)))     # frames sin detección antes de borrar un track
        self.maxlen = int(round(c.window_sec * fps)) + 1             # largo del historial (ventana de movimiento)
        self.cron = Cronometro()
        self.H, self.H_ok = np.eye(3), True                  # última H (diagnóstico)

    @property
    def tiempos_ms(self):
        return self.cron.ms

    @property
    def ids_emitidos(self):
        """IDs creados en total (incluye los que nunca se confirmaron)."""
        return self.nid - 1

    def actualizar(self, frame_bgr, detecciones):
        """frame_bgr: frame ORIGINAL. detecciones: Nx6 [x1,y1,x2,y2,conf,clase] en píxeles del frame original.
        Devuelve la lista de TrackSalida confirmados en este frame."""
        self.cron.inicio()
        # 1. Escala
        if self.escala is None:
            self.escala = Escala(self.cfg, frame_bgr.shape)
        _, gray = self.escala.frame(frame_bgr)
        self.cron.marca("escala")
        # 2. Estabilización
        H, ok = self.estabilizador.paso(gray)
        self.H, self.H_ok = H, ok
        self.cron.marca("estabilizacion")
        # 3. Entrada
        dets = self.entrada.preparar(detecciones, self.escala, gray, H, ok)
        self.cron.marca("entrada")
        # 4-6. Tracking
        confirmados = self.seguir(dets, H, ok)
        self.cron.marca("tracking")
        # 7. ¿Se mueve?
        self.decisor.actualizar(confirmados, ok)
        self.cron.marca("movimiento")
        # 8. Salida
        self.frame += 1
        return a_salida(confirmados, self.escala, self.nombres_clases)

    def seguir(self, dets, H, H_ok):
        """Pasos 4-6 sobre detecciones ya en píxeles de trabajo. Devuelve los Track confirmados de este frame."""
        c = self.cfg
        self._predecir(H, H_ok)                                          # 4
        fuertes = dets[dets[:, 4] >= c.conf_high] if len(dets) else dets
        debiles = dets[(dets[:, 4] >= c.conf_low) & (dets[:, 4] < c.conf_high)] if len(dets) else dets
        pares, sin_det, det_libres = self._asociar(self.tracks, fuertes)  # 5a. ronda fuerte
        for i, j in pares:
            self._corregir(self.tracks[i], fuertes[j])                   # 6
        resto = [self.tracks[i] for i in sin_det]
        pares2, _, _ = self._asociar(resto, debiles)                     # 5b. ronda débil (solo tracks sin pareja)
        for i, j in pares2:
            self._corregir(resto[i], debiles[j])
        con_det = {self.tracks[i].id for i, _ in pares} | {resto[i].id for i, _ in pares2}
        for t in self.tracks:
            if t.id not in con_det:
                t.miss += 1                                              # sigue vivo con su predicción hasta max_miss
        for j in det_libres:
            self.tracks.append(self._nuevo(fuertes[j]))                  # nacen solo con detecciones fuertes
        self.tracks = [t for t in self.tracks if t.miss <= self.max_miss]   # mueren los perdidos
        return [t for t in self.tracks if t.hits >= c.min_hits and t.miss == 0]

    # ---- paso 4 -------------------------------------------------------
    def _predecir(self, H, H_ok):
        """Cada track: historial al frame actual (para el paso 7), Kalman al frame actual con H (ESTABILIZACIÓN) y
        Kalman un frame hacia adelante (movimiento propio). La asociación usa la caja PREDICHA."""
        for t in self.tracks:
            t.warp(H, H_ok)
            m, P = self.kalman.estabilizar(*t.kf, H)
            t.kf = self.kalman.predecir(m, P)
            t.box = xyah_a_caja(t.kf[0][:4])

    # ---- paso 5 -------------------------------------------------------
    def _asociar(self, tracks, dets):
        """IoU (cajas infladas box_pad) + asignación óptima (húngaro). Solo valen parejas con IoU >= match_iou y clase
        compatible. Devuelve (pares (i track, j det), tracks sin pareja, detecciones sin pareja)."""
        if not tracks or len(dets) == 0:
            return [], list(range(len(tracks))), list(range(len(dets)))
        iou = iou_pad(np.array([t.box for t in tracks]), dets[:, :4], self.cfg.box_pad)
        for i, t in enumerate(tracks):
            for j in range(len(dets)):
                if not (t.cls < 0 or dets[j, 5] < 0 or t.cls == int(dets[j, 5])):
                    iou[i, j] = 0                                        # un "car" no se une con un "truck"
        r, cc = linear_sum_assignment(1 - iou)
        pares = [(i, j) for i, j in zip(r, cc) if iou[i, j] >= self.cfg.match_iou]
        mi = {i for i, _ in pares}; mj = {j for _, j in pares}
        return pares, [i for i in range(len(tracks)) if i not in mi], [j for j in range(len(dets)) if j not in mj]

    # ---- paso 6 -------------------------------------------------------
    def _corregir(self, t, det):
        """Kalman corrige con la detección; el track guarda la caja FILTRADA (como Ultralytics) y la clase/conf de la detección."""
        t.kf = self.kalman.corregir(*t.kf, caja_a_xyah(det[:4]))
        f = det.astype(np.float32).copy()
        f[:4] = xyah_a_caja(t.kf[0][:4])
        t.update(f, self.frame)

    def _nuevo(self, det):
        c = self.cfg
        t = Track(self.nid, det, self.frame, self.maxlen, c.cls_min_conf, c.cls_min_hits)
        t.kf = self.kalman.iniciar(caja_a_xyah(det[:4]))
        self.nid += 1
        return t

## PARTE 4 — Decisión de movimiento y salida (igual para los cuatro)
### 4.1 ¿Se mueve o no?
Mide el **desplazamiento neto** del track en 1 s, en coordenadas estabilizadas, con tres defensas:
1. **Rectitud** (`min_straightness`): un objeto real avanza; el ruido de la caja deambula.
2. **Umbral relativo al tamaño** (`rel_move`): frena cajas grandes inestables.
3. **Paralaje:** si el desplazamiento es proporcional al flujo acumulado del suelo, es algo alto y quieto (poste, árbol). Los vehículos no pasan por este filtro.

Histéresis: 3 evaluaciones seguidas para encender, 10 para apagar.

In [ ]:
# ======================================================================
# 4.1 · DecisorMovimiento
# Qué hace : Decide si cada track se mueve DE VERDAD (no por el movimiento del dron)
# ¿Se toca? : No (parámetros en Config → [MOVIMIENTO], [PARALAJE])
# ======================================================================
def ids_sin_paralaje(nombres_clases, cfg):
    """Ids de las clases que no pasan por el filtro de paralaje (vehículos: no sobresalen del suelo como un poste)."""
    return {i for i, n in (nombres_clases or {}).items() if str(n).lower() in cfg.parallax_skip}


class DecisorMovimiento:
    """Mide el desplazamiento NETO del track en la ventana (window_sec) en coordenadas estabilizadas, con tres defensas:
      1) avanzar en línea (min_straightness): el ruido de la caja deambula, un objeto real avanza;
      2) umbral que crece con el tamaño del objeto (rel_move);
      3) paralaje: si el desplazamiento es proporcional al flujo acumulado del suelo, es algo alto y quieto (poste).
    Histéresis: on_frames evaluaciones seguidas para encender, off_frames para apagar."""

    def __init__(self, cfg, fps, skip_ids=()):
        self.cfg = cfg
        self.W = int(round(cfg.window_sec * fps))
        self.thr = cfg.min_move_px
        self.skip_ids = set(skip_ids)

    @staticmethod
    def _rectitud(p):
        """Desplazamiento neto / recorrido, sobre 6 tramos (medianas) para no contar el jitter de la caja."""
        if len(p) < 6:
            return 1.0
        m = np.array([np.median(ch, 0) for ch in np.array_split(p, 6)])
        recorrido = float(np.linalg.norm(np.diff(m, axis=0), axis=1).sum())
        return float(np.linalg.norm(m[-1] - m[0]) / (recorrido + 1e-6))

    def _es_paralaje(self, t, n):
        """Si el objeto está quieto en el mundo, su desplazamiento estabilizado P(t) es proporcional al flujo
        acumulado del suelo en ese punto C(t):  P(t) ≈ k·C(t).  k > 0: sobresale del suelo; k ≈ -1: fijo en la imagen.
        Se exige dirección paralela, k en [k_min, k_max] y buen ajuste en el tiempo (R²)."""
        c = self.cfg
        p, g = t.pts[-n:], t.gs[-n:]
        if len(p) < 6 or t.disp < 1e-6:
            t.k_par = None
            return False
        P = p - p[0]
        C = np.cumsum(g, 0); C = C - C[0]
        Gn = float(np.linalg.norm(C[-1]))
        if Gn < c.parallax_min_ground_px:
            t.k_par = None
            return False
        cos = float(t.vec @ C[-1]) / (t.disp * Gn)
        k = float((P * C).sum() / ((C * C).sum() + 1e-9))
        fit = 1.0 - float(((P - k * C) ** 2).sum()) / (float((P ** 2).sum()) + 1e-9)
        t.k_par = k
        return (abs(cos) >= np.cos(np.deg2rad(c.parallax_angle_deg))
                and c.parallax_k_min <= k <= c.parallax_k_max and fit >= c.parallax_min_fit)

    def actualizar(self, tracks, H_ok):
        c = self.cfg
        if not H_ok:                                                     # sin H fiable no se decide (se mantiene el estado)
            return
        disps = []
        for t in tracks:                                                 # 1) desplazamiento neto en la ventana
            n = min(len(t.pts), self.W + 1)
            if n < max(5, int(0.5 * self.W)):
                t.disp = 0.0; t.vec = np.zeros(2, np.float32); t.par = False; continue
            p = t.pts[-n:]; k = max(1, n // 5)
            t.vec = (np.median(p[-k:], 0) - np.median(p[:k], 0)).astype(np.float32)
            t.disp = float(np.linalg.norm(t.vec))
            disps.append(t.disp)
        self.thr = c.min_move_px
        if c.adaptive and len(disps) >= 5:
            med = np.median(disps); mad = np.median(np.abs(np.array(disps) - med)) * 1.4826
            self.thr = float(min(max(c.min_move_px, med + c.mad_k * mad), c.max_adaptive_px))
        for t in tracks:                                                 # 2) ¿cuenta como movimiento?
            n = min(len(t.pts), self.W + 1)
            if len(t.pts) < max(5, int(0.5 * self.W)):
                continue
            t.straight = self._rectitud(t.pts[-n:])
            size = float(np.sqrt(max((t.box[2] - t.box[0]) * (t.box[3] - t.box[1]), 1.0)))
            thr_t = max(self.thr, c.rel_move * size)
            bw, bh = t.box[2] - t.box[0], t.box[3] - t.box[1]
            estructura = t.cls < 0 and max(bw, bh) / max(min(bw, bh), 1.0) > c.max_unknown_aspect
            raw = t.disp > thr_t and t.straight >= c.min_straightness and not estructura
            t.par = False
            if raw and c.parallax_filter and t.cls not in self.skip_ids and self._es_paralaje(t, n):
                raw, t.par = False, True
                t.par_frames += 1
            if raw:                                                      # 3) histéresis
                t.on_cnt += 1; t.off_cnt = 0
            else:
                t.off_cnt += 1; t.on_cnt = 0
            if not t.moving and t.on_cnt >= c.on_frames:
                t.moving = True; t.toggles += 1
            elif t.moving and t.off_cnt >= c.off_frames:
                t.moving = False; t.toggles += 1
            t.moving_frames += int(t.moving)
            t.max_disp = max(t.max_disp, t.disp)

### 4.2 Salida
`TrackSalida` en píxeles **originales**, `como_array` para benchmarks MOT y `Cronometro` (ms por etapa: `tracker.tiempos_ms`).

In [ ]:
# ======================================================================
# 4.2 · Salida
# Qué hace : Lo que devuelve actualizar(): un TrackSalida por track confirmado, en píxeles del frame ORIGINAL
# ¿Se toca? : No
# ======================================================================
@dataclass
class TrackSalida:
    id: int
    caja: tuple                 # (x1, y1, x2, y2) en píxeles del frame original
    conf: float                 # confianza de la última detección asociada
    clase: int                  # id de clase votado (-1 = desconocido / propuesta)
    nombre_clase: str
    en_movimiento: bool         # decisión de DecisorMovimiento (con histéresis)
    desplazamiento_px: tuple    # desplazamiento neto en la ventana (≈ window_sec), estabilizado, px originales
    paralaje: bool = False      # True = parecía moverse pero lo explica el paralaje (algo alto y quieto)


def a_salida(tracks, escala, nombres_clases):
    nombres = nombres_clases or {}
    out = []
    for t in tracks:
        cls = t.cls
        out.append(TrackSalida(id=int(t.id), caja=tuple(float(x) for x in escala.a_original(t.box)), conf=round(t.conf, 3),
                               clase=int(cls), nombre_clase=nombres.get(cls, "desconocido") if cls >= 0 else "desconocido",
                               en_movimiento=bool(t.moving), desplazamiento_px=tuple(float(x) for x in t.vec / escala.k),
                               paralaje=bool(t.par)))
    return out


def como_array(salida):
    """TrackSalida -> Nx7 [x1, y1, x2, y2, id, conf, clase] (el formato que piden la mayoría de benchmarks MOT)."""
    return np.array([[*s.caja, s.id, s.conf, s.clase] for s in salida], np.float32).reshape(-1, 7)


class Cronometro:
    """Milisegundos de cada etapa del último frame (para saber si llega a tiempo real). tracker.tiempos_ms."""

    def __init__(self):
        self.ms, self._t = {}, None

    def inicio(self):
        self.ms, self._t = {}, time.perf_counter()

    def marca(self, etapa):
        t = time.perf_counter()
        self.ms[etapa] = round(1000 * (t - self._t), 2)
        self._t = t

### 4.3 Dibujo
**Verde** = quieto · **Rojo** = se mueve (con clase) · **Naranja** = se mueve (sin clase) · **Gris** = descartado por paralaje. El recuadro con número es el ID (cada ID, su color: un cambio de color sobre el mismo objeto = cambio de ID).

In [ ]:
# ======================================================================
# 4.3 · dibujar
# Qué hace : Dibuja la salida sobre el frame original (solo para ver; no afecta al tracking)
# ¿Se toca? : No
# ======================================================================
VERDE, ROJO, NARANJA, GRIS = (60, 200, 60), (40, 40, 255), (0, 150, 255), (170, 170, 170)


def color_id(tid):
    """Color estable por ID: un cambio de color sobre el mismo objeto = el tracker le cambió el ID."""
    c = cv2.cvtColor(np.uint8([[[int((tid * 47) % 180), 220, 230]]]), cv2.COLOR_HSV2BGR)[0, 0]
    return int(c[0]), int(c[1]), int(c[2])


def dibujar(frame, salida, titulo=None):
    """Verde = quieto · Rojo = se mueve (con clase) · Naranja = se mueve (sin clase) · Gris = descartado por paralaje.
    El recuadro con número es el ID (cada ID, su color)."""
    for s in salida:
        if not s.en_movimiento and s.clase < 0 and not s.paralaje:
            continue                                                     # propuesta quieta: no aporta nada
        color = GRIS if (s.paralaje and not s.en_movimiento) else (VERDE if not s.en_movimiento else (ROJO if s.clase >= 0 else NARANJA))
        x1, y1, x2, y2 = (int(round(v)) for v in s.caja)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        txt = f"{s.id}"
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        cv2.rectangle(frame, (x1, max(0, y1 - th - 8)), (x1 + tw + 8, max(th + 8, y1)), color_id(s.id), -1)
        cv2.putText(frame, txt, (x1 + 4, max(th + 4, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, s.nombre_clase, (x1 + tw + 12, max(14, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    n_mov = sum(s.en_movimiento for s in salida)
    cv2.putText(frame, f"{titulo + ' | ' if titulo else ''}en movimiento: {n_mov} | tracks: {len(salida)}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    return frame

## PARTE 5 — Detector externo y bucle en tiempo real
### 5.1 Detector externo
**Entra:** un frame BGR original. **Sale:** `Nx6 [x1, y1, x2, y2, conf, clase]` en píxeles de **ese** frame, y `nombres = {id: "car", ...}`.

Aquí se simula con YOLO, con los mismos modelos y umbrales de la comparativa: OBB (DOTA) para tomas cenitales y COCO para oblicuas. **En el otro entorno solo se reemplaza esta celda.** El umbral de confianza del detector debe quedar **por debajo** de `conf_low` (0.10) o por lo menos no muy por encima: el tracker hace una 2ª ronda con las detecciones débiles.

In [ ]:
# ======================================================================
# 5.1 · Detector externo (simulado con YOLO)
# Qué hace : Devuelve las detecciones de un frame en el formato que espera el tracker
# ¿Se toca? : SÍ — en el otro entorno, esta celda se sustituye por el detector que ya existe
# ======================================================================
from ultralytics import YOLO

class DetectorYOLO:
    def __init__(self, pesos, clases, imgsz=1280, conf=0.15):
        self.modelo = YOLO(pesos)
        inv = {str(v).lower(): k for k, v in self.modelo.names.items()}
        self.clases = [inv[c] for c in clases]
        self.nombres = dict(self.modelo.names)
        self.imgsz, self.conf = imgsz, conf

    def __call__(self, frame):
        r = self.modelo.predict(frame, imgsz=self.imgsz, conf=self.conf, classes=self.clases, verbose=False)[0]
        b = r.obb if getattr(r, "obb", None) is not None else r.boxes     # OBB: se usa su caja horizontal
        if b is None or len(b) == 0:
            return np.zeros((0, 6), np.float32)
        return np.concatenate([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()[:, None], b.cls.cpu().numpy()[:, None]], 1).astype(np.float32)

DETECTOR_CENITAL = lambda: DetectorYOLO("yolo11n-obb.pt", ("small vehicle", "large vehicle"), conf=0.15)
DETECTOR_OBLICUO = lambda: DetectorYOLO("yolo11n.pt", ("car", "truck", "bus", "motorcycle"), conf=0.25)

### 5.2 Bucle frame a frame
Lee un frame, detecta, actualiza el tracker, dibuja y escribe: lo mismo que haría el otro entorno con la cámara en vivo.

Por cada frame guarda los ms del detector y los de cada etapa del tracker (`tracker.tiempos_ms`).

**Salidas** en `salidas/tiempo_real/`:
- `<video>_<tracker>.mp4`: video anotado.
- `<video>_<tracker>_tracks.csv`: tracks en formato MOT (`frame` empieza en 1), para el benchmark.

In [ ]:
# ======================================================================
# 5.2 · procesar
# Qué hace : Bucle frame a frame (detector -> tracker -> dibujo) con medición de tiempos
# ¿Se toca? : No (el tracker se crea con crear_tracker, en el Paso B)
# ======================================================================
def procesar(video, detector, crear_tracker, out_dir="salidas/tiempo_real", max_frames=None, out_width=1920):
    cap = cv2.VideoCapture(str(video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    tracker = crear_tracker(fps, detector.nombres)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    base = Path(out_dir) / f"{Path(video).stem}_{tracker.nombre}"
    writer, filas, mot = None, [], []
    k = 0
    while max_frames is None or k < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        t0 = time.perf_counter()
        dets = detector(frame)                                           # detector externo
        t_det = 1000 * (time.perf_counter() - t0)
        tracks = tracker.actualizar(frame, dets)                         # <- el tracker
        fila = dict(frame=k, detector_ms=round(t_det, 1), tracker_ms=round(sum(tracker.tiempos_ms.values()), 2),
                    **{f"{e}_ms": v for e, v in tracker.tiempos_ms.items()}, tracks=len(tracks),
                    en_movimiento=sum(t.en_movimiento for t in tracks))
        filas.append(fila)
        mot += [[k + 1, *r] for r in como_array(tracks)]                 # formato MOT: frame, x1, y1, x2, y2, id, conf, clase
        vis = dibujar(frame.copy(), tracks, tracker.nombre)
        if out_width:
            vis = cv2.resize(vis, (out_width, int(round(vis.shape[0] * out_width / vis.shape[1]))), interpolation=cv2.INTER_AREA)
        if writer is None:
            writer = cv2.VideoWriter(f"{base}.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (vis.shape[1], vis.shape[0]))
        writer.write(vis)
        k += 1
    cap.release()
    if writer is not None:
        writer.release()
    pd.DataFrame(mot, columns=["frame", "x1", "y1", "x2", "y2", "id", "conf", "clase"]).to_csv(f"{base}_tracks.csv", index=False)
    df = pd.DataFrame(filas)
    print(f"{tracker.nombre}: {k} frames | tracker {df.tracker_ms.median():.1f} ms/frame (p95 {df.tracker_ms.quantile(0.95):.1f}) | "
          f"detector {df.detector_ms.median():.0f} ms/frame | IDs creados: {getattr(tracker, 'ids_emitidos', 'n/a')} | {base}.mp4")
    return df, tracker

## PARTE 6 — Ejecución
`PARTE 1` → `PARTE 5` solo definen cosas; aquí se ejecuta.

### Paso A — Pruebas (ejecutar primero, sobre todo en un entorno nuevo)
No necesitan video ni detector: comprueban cada pieza con datos sintéticos.

In [ ]:
# ======================================================================
# Paso A · Pruebas
# Qué hace : Comprueba la estabilización del Kalman y que un objeto quieto con el dron en paneo conserve su ID
# ¿Se toca? : No — se ejecuta en el Paso A
# ======================================================================
def selftest():
    kf = KalmanXYAH()
    m, P = kf.iniciar(np.array([100.0, 50.0, 0.5, 20.0]))
    m[4:6] = [3.0, 0.0]                                                  # el objeto avanza 3 px/frame en x
    mt, _ = KalmanXYAH.estabilizar(m, P, np.array([[1, 0, 10], [0, 1, -5], [0, 0, 1]], np.float64))
    assert np.allclose(mt[:2], [110, 45]) and np.allclose(mt[4:6], [3, 0]), "traslación"
    mz, _ = KalmanXYAH.estabilizar(m, P, np.diag([2.0, 2.0, 1.0]))
    assert np.allclose(mz[:4], [200, 100, 0.5, 40]) and np.allclose(mz[4:6], [6, 0]), "zoom"
    trk = TrackerPropio(30.0, ConfigPropio(use_proposals=False))
    for f in range(60):                                                  # objeto QUIETO en el suelo, paneo de 14 px/frame
        x = 200 + 14 * f
        d = np.array([[x, 100, x + 12, 112, 0.9, 0]], np.float32)
        Hf = np.array([[1, 0, 14 if f else 0], [0, 1, 0], [0, 0, 1]], np.float64)
        out = trk.seguir(d, Hf, True); trk.frame += 1
    assert trk.ids_emitidos == 1 and len(out) == 1, trk.ids_emitidos
    print("tracker_propio OK | estabilización del Kalman (traslación y zoom) | objeto quieto, paneo de 14 px/frame -> 1 ID")


def prueba_comun():
    """Escala ida y vuelta, aviso de formato [x, y, w, h] y detecciones vacías."""
    esc = Escala(Config(), (2160, 3840, 3))                              # frame 4K -> trabajo 960 x 540
    assert (esc.ww, esc.wh) == (960, 540)
    d = np.array([[400, 400, 480, 440, 0.9, 2]], np.float32)
    assert np.allclose(esc.a_original(esc.a_trabajo(d)[0, :4]), d[0, :4]), "ida y vuelta"
    try:
        EntradaDetecciones.revisar(np.array([[400, 400, 80, 40, 0.9, 2]]))     # [x, y, w, h] por error
        raise AssertionError("debió avisar de cajas al revés")
    except ValueError:
        pass
    assert EntradaDetecciones.revisar(None).shape == (0, 6) and EntradaDetecciones.revisar([]).shape == (0, 6)
    assert como_array([]).shape == (0, 7)
    print("común OK | escala ida y vuelta, aviso de formato [x, y, w, h], detecciones vacías")


def prueba_pipeline():
    """Pipeline completo con un video sintético: un auto cruza sobre fondo con textura y la cámara hace paneo.
    Debe salir 1 ID, en movimiento, con la caja en píxeles originales."""
    rng = np.random.default_rng(0)
    fondo = cv2.GaussianBlur((rng.random((1200, 4000)) * 255).astype(np.uint8), (7, 7), 0)
    fondo = cv2.cvtColor(fondo, cv2.COLOR_GRAY2BGR)
    W, H = 1920, 1080
    trk = TrackerPropio(30.0, nombres_clases={2: "car"})
    for f in range(90):
        cam_x = 4 * f                                                    # la cámara se desplaza 4 px/frame
        frame = fondo[60:60 + H, cam_x:cam_x + W].copy()
        x = 300 + 12 * f - cam_x                                         # el auto avanza 12 px/frame en el suelo
        cv2.rectangle(frame, (x, 500), (x + 80, 540), (0, 0, 255), -1)
        s = trk.actualizar(frame, np.array([[x, 500, x + 80, 540, 0.9, 2]], np.float32))
    assert len(s) == 1 and s[0].en_movimiento and s[0].nombre_clase == "car", s
    assert abs(s[0].caja[0] - x) < 6, (s[0].caja, x)
    print(f"pipeline OK | 1 ID tras 90 frames, en movimiento, caja en píxeles originales | ms por frame: {trk.tiempos_ms}")


prueba_comun()
selftest()
prueba_pipeline()

### Paso B — Procesar un video

In [ ]:
# ======================================================================
# Paso B · Procesar un video
# Qué hace : Crea el tracker y procesa el video frame a frame
# ¿Se toca? : SÍ — VIDEO_PATH, detector y crear_tracker. Para probar rápido: max_frames=300
# ======================================================================
VIDEO_PATH = Path("/Users/Alex/Documents/videos_VIGIA/vigia_8.mp4")     # <-- CAMBIAR por tu video
detector = DETECTOR_OBLICUO()                                          # oblicua: DETECTOR_OBLICUO() | cenital: DETECTOR_CENITAL()
def crear_tracker(fps, nombres):
    cfg = ConfigPropio(work_width=960, use_proposals=True)              # benchmark MOT: use_proposals=False
    return TrackerPropio(fps, cfg, nombres)

df, tracker = procesar(VIDEO_PATH, detector, crear_tracker, max_frames=None)
display(df.describe().T[["50%", "max"]].rename(columns={"50%": "mediana"}).round(2))

In [ ]:
display(Video(f"salidas/tiempo_real/{VIDEO_PATH.stem}_{tracker.nombre}.mp4", embed=False, width=1100))

### Paso C — ¿Llega a tiempo real?
ms por etapa (mediana y p95) del tracker, **sin** contar el detector. A 30 fps hay 33 ms por frame **para todo**, detector incluido. Si no llega: `use_proposals=False` y/o un `work_width` menor.

In [ ]:
etapas = [c for c in df.columns if c.endswith("_ms")]
display(df[etapas].describe(percentiles=[0.5, 0.95]).T[["50%", "95%", "max"]].round(2))